In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('/content/anime.csv')

In [3]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
df.shape

(12294, 7)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [6]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [7]:
print("\n--- Missing Values Before Cleaning ---")
print(df.isnull().sum())


--- Missing Values Before Cleaning ---
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64


In [8]:
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

In [9]:
print("\n--- Missing Values After Cleaning ---")
print(df.isnull().sum())


--- Missing Values After Cleaning ---
anime_id     0
name         0
genre       62
type        25
episodes     0
rating       0
members      0
dtype: int64


In [10]:
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna('Unknown')

In [11]:
print("\n--- Missing Values After Cleaning ---")
print(df.isnull().sum())


--- Missing Values After Cleaning ---
anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64


In [15]:
from sklearn.preprocessing import StandardScaler

print("Step 1: Select Features for Similarity")
# Choose relevant features for computing similarity (combining numerical and categorical attributes)
selected_features = ['genre', 'type', 'rating', 'members']

# Create a working subset dataframe
df_features = df[selected_features].copy()
print(f"Selected features: {selected_features}")


print("\n Step 2: Convert Categorical Features into Numerical Representations ")
df_encoded = pd.get_dummies(df_features, columns=['type'], drop_first=False)

# For standard categorical conversion, we can also binarize/dummy-encode genre if needed:
df_encoded = pd.get_dummies(df_encoded, columns=['genre'], drop_first=False)

print(f"Shape after categorical conversion/encoding: {df_encoded.shape}")


print("\n Step 3: Normalize Numerical Features ")
# Identify numerical columns that need normalization (e.g., rating, members)
numerical_cols = ['rating', 'members']

# Initialize StandardScaler to scale features to mean=0 and std=1
scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

print("\nFeature Extraction & Normalization Complete! First 5 rows of processed feature matrix:")
display(df_encoded.head())

Step 1: Select Features for Similarity
Selected features: ['genre', 'type', 'rating', 'members']

 Step 2: Convert Categorical Features into Numerical Representations 
Shape after categorical conversion/encoding: (12294, 3274)

--- Step 3: Normalize Numerical Features ---

Feature Extraction & Normalization Complete! First 5 rows of processed feature matrix:


,rating,members,type_Movie,type_Music,type_ONA,type_OVA,type_Special,type_TV,type_Unknown,genre_Action,...,"genre_Slice of Life, Space","genre_Slice of Life, Supernatural",genre_Space,genre_Sports,"genre_Super Power, Supernatural, Vampire",genre_Supernatural,genre_Thriller,genre_Unknown,genre_Vampire,genre_Yaoi
0,2.845534,3.330241,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2.737388,14.148406,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2.727556,1.754713,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2.648904,11.957666,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2.639073,2.429742,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Compute the Cosine Similarity Matrix
print("Computing Cosine Similarity matrix...")
cosine_sim_matrix = cosine_similarity(df_encoded, df_encoded)
print(f"Cosine Similarity Matrix shape: {cosine_sim_matrix.shape}")


# Step 2: Design the Recommendation Function
def recommend_anime(anime_name, similarity_matrix=cosine_sim_matrix, df_source=df, top_n=5, threshold=0.5):

    # Check if the anime exists in the dataset
    matching_indices = df_source[df_source['name'].str.lower() == anime_name.lower()].index

    if len(matching_indices) == 0:
        print(f"Sorry, '{anime_name}' was not found in the dataset.")
        return pd.DataFrame()

    idx = matching_indices[0]

    # Get pairwise similarity scores for the target anime with all other anime
    sim_scores = list(enumerate(similarity_matrix[idx]))

    # Sort the anime based on similarity scores in descending order (excluding itself at index 0)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Filter by threshold and take top_n recommendations
    filtered_scores = [item for item in sim_scores[1:] if item[1] >= threshold]
    top_recommendations = filtered_scores[:top_n]

    # Extract indices and scores
    anime_indices = [i[0] for i in top_recommendations]
    scores = [i[1] for i in top_recommendations]

    # Build recommendation dataframe
    rec_df = df_source.iloc[anime_indices].copy()
    rec_df['similarity_score'] = scores

    print(f"\n--- Recommendations for '{df_source.iloc[idx]['name']}' (Threshold >= {threshold}) ---")
    return rec_df[['name', 'genre', 'type', 'rating', 'similarity_score']]


# Step 3: Test the Recommendation Function with an Example
target_anime = 'Kimi no Na wa.'
recommendations = recommend_anime(target_anime, top_n=5, threshold=0.3)
display(recommendations)


# Step 4: Experiment with Different Threshold Values
print("\n" + "="*50)
print("--- Experimenting with Threshold Values ---")
print("="*50)
for thresh in [0.2, 0.5, 0.7]:
    res = recommend_anime('Steins;Gate', top_n=3, threshold=thresh)
    print(f"Threshold: {thresh} -> Returned {len(res)} recommendations.")

Computing Cosine Similarity matrix...
Cosine Similarity Matrix shape: (12294, 12294)

--- Recommendations for 'Kimi no Na wa.' (Threshold >= 0.3) ---


,name,genre,type,rating,similarity_score
18,Ookami Kodomo no Ame to Yuki,"Fantasy, Slice of Life",Movie,8.84,0.942224
59,Steins;Gate Movie: Fuka Ryouiki no Déjà vu,"Sci-Fi, Thriller",Movie,8.61,0.938782
25,Suzumiya Haruhi no Shoushitsu,"Comedy, Mystery, Romance, School, Sci-Fi, Supe...",Movie,8.81,0.938499
60,Hotarubi no Mori e,"Drama, Romance, Shoujo, Supernatural",Movie,8.61,0.938446
71,Hotaru no Haka,"Drama, Historical",Movie,8.58,0.937622



--- Experimenting with Threshold Values ---

--- Recommendations for 'Steins;Gate' (Threshold >= 0.2) ---
Threshold: 0.2 -> Returned 3 recommendations.

--- Recommendations for 'Steins;Gate' (Threshold >= 0.5) ---
Threshold: 0.5 -> Returned 3 recommendations.

--- Recommendations for 'Steins;Gate' (Threshold >= 0.7) ---
Threshold: 0.7 -> Returned 3 recommendations.


In [ ]:
#User-Based Collaborative Filtering (User-User):
#It finds users who have similar rating patterns or preferences to you . If User A and User B both liked anime X, Y, and Z, the system assumes User A will also enjoy whatever else User B likes.
#Pros/Cons: Great for capturing dynamic shifts in user interests, but computationally expensive when the user base grows massively (since the number of users is often much larger than the number of items).

In [ ]:
#item-Based Collaborative Filtering (Item-Item):
#It looks at the relationships between items rather than users. If a large group of people who liked Anime X also gave high ratings to Anime Y, the system determines that Anime X and Anime Y are similar. If you liked Anime X, it recommends Anime Y.
#Pros/Cons: Items change much less frequently than user preferences, making item-item similarity matrices pre-computable and much more scalable for large platforms (like Amazon or Netflix).

In [ ]:
#Collaborative filtering is a technique used by recommendation systems to predict a user's interests or preferences by collecting and analyzing rating patterns, purchase history, or behavior data from many users.

In [ ]:
#Working:
#Data Collection (The Interaction Matrix): It builds a large matrix where rows represent users, columns represent items , and values represent ratings or interactions (e.g., 1 to 5 stars, clicks, or watches).
#Pattern Recognition / Neighborhood Finding: The algorithm looks for mathematical patterns in this matrix. It identifies groups of users with similar tastes (user-based) or items that tend to be consumed together (item-based).
#Prediction & Recommendation: Based on these learned patterns, it predicts what rating or score a target user would give to an item they haven't interacted with yet, and recommends the items with the highest predicted scores.